[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter7/crewai-multi-agent.ipynb)

# Chapter 7: Multi-Agent Systems with CrewAI

This notebook demonstrates building collaborative multi-agent systems using CrewAI, where specialized agents work together to accomplish complex tasks.

**What you'll learn:**
- Define specialized agents with distinct roles, goals, and backstories
- Create tasks with dependencies using CrewAI's context system
- Execute a sequential multi-agent workflow (research → writing)

**Prerequisites:** `pip install crewai` and set `OPENAI_API_KEY`.

## Setup

We import CrewAI's core components for defining agents, tasks, and orchestrating multi-agent workflows.

In [1]:
import os
from crewai import Agent, Task, Crew, Process

## Define Agents

Each agent gets a role, goal, and backstory that shapes how the LLM behaves — here we create a researcher to find trends and a writer to turn them into content.

In [2]:
# Agent 1: Market Research Analyst
researcher = Agent(
    role="Senior Market Research Analyst",
    goal="Find groundbreaking and emerging trends in the field of Artificial Intelligence.",
    backstory=(
        "You are an expert market research analyst with a keen eye for emerging technological trends. "
        "You continuously scan the horizon for a major technological shift, and you have a deep understanding "
        "of the AI landscape. Your goal is to identify high-potential topics that are newsworthy and relevant to a tech audience."
    ),
    verbose=True,
    allow_delegation=False,
)

# Agent 2: Technology Content Strategist
writer = Agent(
    role="Technology Content Strategist",
    goal="Craft a compelling and informative blog post based on research findings.",
    backstory=(
        "You are a renowned content strategist known for simplifying complex technological topics "
        "into engaging narratives. You take raw data and research insights and transform them into "
        "high-quality articles that resonate with both technical experts and curious beginners. "
        "Your writing style is clear, insightful, and accessible."
    ),
    verbose=True,
    allow_delegation=False,
)

## Define Tasks

Tasks describe the work each agent must do; the `context` parameter lets a downstream task receive the output of an earlier one, creating a dependency chain.

In [3]:
# Task 1: Research AI Trends
# The 'agent' parameter assigns this task to the 'researcher' agent.
research_task = Task(
    description=(
        "Identify and analyze the top 3 most significant emerging trends in AI for the current year. "
        "Focus on trends related to large language models (LLMs), generative AI, and real-world applications. "
        "Provide a summary for each trend, highlighting its potential impact and key players."
    ),
    expected_output=(
        "A detailed report containing three emerging AI trends. Each trend section must include: "
        "1. Trend Title. "
        "2. Concise summary (2-3 sentences). "
        "3. Explanation of potential market impact. "
        "4. Key companies or research labs involved."
    ),
    agent=researcher
)

In [4]:
# Task 2: Write Blog Post
# This task depends on the output of 'research_task'. We define this dependency
# using the 'context' parameter. The crew will ensure 'research_task' completes
# first and its output is available to 'writer_task'.
writer_task = Task(
    description=(
        "Using the research findings provided as context, write an engaging blog post suitable for a general tech audience. "
        "The post should be approximately 500 words long. "
        "Structure the post with an introduction, sections for each of the three trends, and a concluding paragraph."
    ),
    expected_output=(
        "A well-structured and polished blog post of at least 500 words, "
        "formatted in markdown. The post must be engaging and easy to understand for non-experts."
    ),
    agent=writer,
    context=[research_task]
)

## Run the Crew

The Crew assembles agents and tasks into a sequential pipeline, then `kickoff()` executes them in order — research first, then writing.

In [5]:
# Create the crew and define the process.
# Process.sequential means tasks will be executed one after another in the order they appear in the list.
ai_trends_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writer_task],
    process=Process.sequential,
    verbose=True,
)

# --- Execution ---

# Start the crew's work. The input dictionary can provide initial data for the first task.
result = ai_trends_crew.kickoff(inputs={'topic': 'AI trends'})

print("\n\n##############################")
print("## Crew Execution Finished! ##")
print("##############################\n")
print("Final Blog Post:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  fa9926e3-5e1e-48f4-8dbf-e7deef46a9a0                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Identify and analyze the top 3 most significant emerging trends in AI for the current year. Focus on     │
│  trends related to large language models (LLMs), generative AI, and real-world applications. Provide a summary  │
│  for each trend, highlighting its potential impact and key players.                                             │
│  ID: 005457e5-1fdc-4af1-b82e-3f56dfc4b80c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Market Research Analyst                                                                          │
│                                                                                                                 │
│  Task: Identify and analyze the top 3 most significant emerging trends in AI for the current year. Focus on     │
│  trends related to large language models (LLMs), generative AI, and real-world applications. Provide a summary  │
│  for each trend, highlighting its potential impact and key players.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Market Research Analyst                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Emerging AI Trends Report 2024: Focus on LLMs, Generative AI, and Real-World Applications                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### 1. Trend Title: **Multimodal Large Language Models (LLMs) Integration**                                   │
│                                                                                                                 │
│  **Summary:**                                                                                                   │
│  The next wave of large language models is increasingly multimodal, meaning they can process and generate not   │
│  only text but also images, audio, and video. These models combine natural language understanding with other    │
│  sensory inputs to produce more coherent, contextually rich outputs and enable seamless human-computer          │
│  interaction across multiple content types.                                                                     │
│                                                                                                                 │
│  **Potential Market Impact:**                                                                                   │
│  Multimodal LLMs promise to revolutionize industries such as content creation, entertainment, education, and    │
│  healthcare by providing versatile, context-aware AI assistants capable of tasks ranging from real-time video   │
│  captioning to interactive tutoring and design generation. This trend is expected to catalyze the next          │
│  generation of digital assistants, reducing the need for multiple specialized tools and enhancing productivity  │
│  through unified AI interfaces. Enterprises could see significant cost savings and innovation acceleration by   │
│  adopting multifunctional AI platforms.                                                                         │
│                                                                                                                 │
│  **Key Companies / Research Labs:**                                                                             │
│  - **OpenAI** (GPT-4 with vision capabilities)                                                                  │
│  - **DeepMind** (Flamingo model)                                                                                │
│  - **Google DeepMind / Google Research** (PaLM-E, Pathways)                                                     │
│  - **Meta AI** (Make-A-Scene and other multimodal models)                                                       │
│  - **Anthropic** (Claude with multimodal capabilities in development)                                           │
│                                                                                                                 │
│  ---                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Identify and analyze the top 3 most significant emerging trends in AI for the current year. Focus on trends    │
│  related to large language models (LLMs), generative AI, and real-world applications. Provide a summary for     │
│  each trend, highlighting its potential impact and key players.                                                 │
│  Agent:                                                                                                         │
│  Senior Market Research Analyst                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technology Content Strategist                                                                           │
│                                                                                                                 │
│  Task: Using the research findings provided as context, write an engaging blog post suitable for a general      │
│  tech audience. The post should be approximately 500 words long. Structure the post with an introduction,       │
│  sections for each of the three trends, and a concluding paragraph.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technology Content Strategist                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Emerging AI Trends in 2024: Multimodal LLMs, Customization, and Synthetic Data                               │
│                                                                                                                 │
│  Artificial intelligence continues to evolve at a breathtaking pace, reshaping how we interact with technology  │
│  and making previously impossible tasks routine. As we advance through 2024, three major AI trends are          │
│  standing out, promising to revolutionize industries from healthcare to entertainment: multimodal large         │
│  language models (LLMs), democratization of foundation model customization, and the growing role of generative  │
│  AI in synthetic data and digital twins. Let’s dive into each of these trends to understand what they mean and  │
│  how they’re transforming the AI landscape.                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Multimodal Large Language Models (LLMs) Integration                                                      │
│                                                                                                                 │
│  Traditionally, large language models have focused on understanding and generating text. The exciting           │
│  evolution in 2024 is that these models are becoming *multimodal* — capable of comprehending and producing not  │
│  just text but images, audio, and video as well. Imagine an AI assistant that can watch a video clip, analyze   │
│  its content, generate a summary, and even create a related image, all seamlessly within the same               │
│  conversation.                                                                                                  │
│                                                                                                                 │
│  The integration of multiple sensory inputs enables AI systems to offer richer, more context-aware outputs.     │
│  This evolution enhances human-computer interaction and paves the way for breakthroughs across sectors:         │
│                                                                                                                 │
│  - **Content creation:** Automated multimedia content generation, from videos with dynamic subtitles to         │
│  interactive storyboards.                                                                                       │
│  - **Education:** AI tutors that can analyze spoken questions, interpret diagrams, and provide tailored visual  │
│  explanations.                                                                                                  │
│  - **Healthcare:** Systems that combine doctor's notes, medical images, and patient speech for better           │
│  diagnostics and recommendations.                                                                               │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings provided as context, write an engaging blog post suitable for a general tech       │
│  audience. The post should be approximately 500 words long. Structure the post with an introduction, sections   │
│  for each of the three trends, and a concluding paragraph.                                                      │
│  Agent:                                                                                                         │
│  Technology Content Strategist                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



##############################
## Crew Execution Finished! ##
##############################

Final Blog Post:
```markdown
# Emerging AI Trends in 2024: Multimodal LLMs, Customization, and Synthetic Data

Artificial intelligence continues to evolve at a breathtaking pace, reshaping how we interact with technology and making previously impossible tasks routine. As we advance through 2024, three major AI trends are standing out, promising to revolutionize industries from healthcare to entertainment: multimodal large language models (LLMs), democratization of foundation model customization, and the growing role of generative AI in synthetic data and digital twins. Let’s dive into each of these trends to understand what they mean and how they’re transforming the AI landscape.

---

## 1. Multimodal Large Language Models (LLMs) Integration

Traditionally, large language models have focused on understanding and generating text. The exciting evolution in 2024 is that these models are becomi